In [0]:
dbutils.fs.ls("/")
 


[FileInfo(path='abfss://bronze@databricksexternallej001.dfs.core.windows.net/Open Transaction DataFULL.csv', name='Open Transaction DataFULL.csv', size=131766890, modificationTime=1775484335000)]

In [0]:

tiers = ["bronze","silver","gold"]
adls_path = {tier:f"abfss://{tier}@databricksexternallej001.dfs.core.windows.net/" for tier in tiers }
#dbutils.fs.ls("abfss://bronze@databricksexternallej001.dfs.core.windows.net/")

bronze_path = adls_path["bronze"]
dbutils.fs.ls(bronze_path)

# Accessing paths
bronze_adls = adls_path["bronze"]
silver_adls = adls_path["silver"]
gold_adls = adls_path["gold"] 


In [0]:

import requests
import json
from datetime import date, timedelta

In [0]:
# Remove this before running Data Factory Pipeline
start_date = date.today() - timedelta(1)
end_date = date.today()

In [0]:
# Construct the API URL with start and end dates provided by Data Factory, formatted for geojson output.
url = f"https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime={start_date}&endtime={end_date}"

try:
    # Make the GET request to fetch data
    response = requests.get(url)

    # Check if the request was successful
    response.raise_for_status()  # Raise HTTPError for bad responses (4xx or 5xx)
    data = response.json().get('features', [])

    if not data:
        print("No data returned for the specified date range.")
    else:
        # Specify the ADLS path
        file_path = f"{bronze_adls}/{start_date}_earthquake_data.json"

        # Save the JSON data
        json_data = json.dumps(data, indent=4)
        dbutils.fs.put(file_path, json_data, overwrite=True)
        print(f"Data successfully saved to {file_path}")
except requests.exceptions.RequestException as e:
    print(f"Error fetching data from API: {e}")

Wrote 347585 bytes.
Data successfully saved to abfss://bronze@databricksexternallej001.dfs.core.windows.net//2026-04-06_earthquake_data.json


In [0]:
data[0]

{'type': 'Feature',
 'properties': {'mag': 1.7,
  'place': '33 km N of Benton, California',
  'time': 1775519992301,
  'updated': 1775520130955,
  'tz': None,
  'url': 'https://earthquake.usgs.gov/earthquakes/eventpage/nn00913838',
  'detail': 'https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=nn00913838&format=geojson',
  'felt': None,
  'cdi': None,
  'mmi': None,
  'alert': None,
  'status': 'automatic',
  'tsunami': 0,
  'sig': 44,
  'net': 'nn',
  'code': '00913838',
  'ids': ',nn00913838,',
  'sources': ',nn,',
  'types': ',origin,phase-data,',
  'nst': 9,
  'dmin': 0.127,
  'rms': 0.3193,
  'gap': 88,
  'magType': 'ml',
  'type': 'earthquake',
  'title': 'M 1.7 - 33 km N of Benton, California'},
 'geometry': {'type': 'Point', 'coordinates': [-118.4922, 38.1246, 6.898]},
 'id': 'nn00913838'}

In [0]:
# Define your variables
output_data = {
    "start_date": start_date.isoformat(),
    "end_date": end_date.isoformat(),
    "bronze_adls": bronze_adls,
    "silver_adls": silver_adls,
    "gold_adls": gold_adls
}

# Serialize the dictionary to a JSON string
output_json = json.dumps(output_data)

# Log the serialized JSON for debugging
print(f"Serialized JSON: {output_json}")

# Return the JSON string
dbutils.notebook.exit(output_json)

dbutils.jobs.taskValues.set(key = "bronze_output", value = output_data)